# 🏏 CricketIQ 2.0 — Notebook 04: Player Feature Engineering & Bayesian Shrinkage
## BCA Machine Learning Project Pipeline — Phase 4

**Objective:**
1. Formulate empirical Bayesian shrinkage priors to prevent low-sample distortion for debutants and newcomers.
2. Compute granular phase-by-phase capabilities (Powerplay, Middle, Death overs).
3. Generate career ball-by-ball profiles across all active IPL players.


### Mathematical Formulation of Bayesian Priors

$$\text{Average}_{\text{shrunk}} = \frac{\text{Runs} + (\mu_{\text{prior}} \times w_{\text{prior}})}{\text{Dismissals} + w_{\text{prior}}}$$

$$\text{Strike Rate}_{\text{shrunk}} = \frac{\text{Runs} + (\text{SR}_{\text{prior}} / 100 \times w_{\text{prior}})}{\text{Balls} + w_{\text{prior}}} \times 100$$

$$\text{Economy}_{\text{shrunk}} = \frac{\text{Runs Conceded} + (\text{Econ}_{\text{prior}} / 6 \times w_{\text{prior}})}{\text{Balls Bowled} + w_{\text{prior}}} \times 6$$

Where league benchmarks:
* $\mu_{\text{prior}} = 22.0$ runs/dismissal ($w=5$)
* $\text{SR}_{\text{prior}} = 125.0$ runs/100 balls ($w=30$)
* $\text{Econ}_{\text{prior}} = 8.50$ runs/over ($w=24$)


In [1]:
import sys
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve() if Path("..").resolve().name == "CricketIQ 2.0" else Path(".").resolve()
sys.path.insert(0, str(BASE_DIR))

from src.data_loader import load_data
from src.player_features import build_global_player_stats

matches, deliveries = load_data(BASE_DIR)
global_player_stats = build_global_player_stats(matches, deliveries)

print(f"✅ Compiled full ball-by-ball profiles for {len(global_player_stats):,} IPL players.")

sample_players = ['V Kohli', 'JJ Bumrah', 'VR Iyer', 'Mohammed Siraj', 'PD Salt', 'Rashid Khan']
rows = []
for p in sample_players:
    st = global_player_stats.get(p, {})
    rows.append({
        'Player': p,
        'Role': 'BAT' if st.get('is_batter') and not st.get('is_bowler') else ('BOWL' if st.get('is_bowler') and not st.get('is_batter') else 'ALL'),
        'Innings': st.get('bat_innings', 0),
        'Runs': st.get('bat_runs', 0),
        'Shrunk Avg': round(st.get('bat_avg', 0), 1),
        'Shrunk SR': round(st.get('bat_sr', 0), 1),
        'Wickets': st.get('bowl_wickets', 0),
        'Shrunk Econ': round(st.get('bowl_econ', 0), 2),
        'Perf Score (⚡%)': f"{st.get('perf_score', 0)}%"
    })

display(pd.DataFrame(rows))


✅ Compiled full ball-by-ball profiles for 731 IPL players.


,Player,Role,Innings,Runs,Shrunk Avg,Shrunk SR,Wickets,Shrunk Econ,Perf Score (⚡%)
0,V Kohli,ALL,241,7987,38.9,128.3,4,8.84,78%
1,JJ Bumrah,ALL,26,68,16.2,95.9,168,7.50,90%
2,VR Iyer,ALL,49,1326,30.6,132.4,3,10.23,61%
3,Mohammed Siraj,ALL,20,109,15.6,96.4,93,8.86,61%
4,PD Salt,BAT,21,653,31.8,166.4,0,8.50,84%
5,Rashid Khan,ALL,58,545,15.6,151.3,149,6.99,81%
